# test_dense_retriever.py

In [1]:
import sys

sys.path.insert(
    0,
    r"C:\Users\User\Nasa Rag SYS\rag\retrieval"
)

from dense_retriever import DenseRetriever


def test_dense_retriever():

    retriever = DenseRetriever()

    query = (
        "How does oxygen concentration affect "
        "flame behavior in microgravity?"
    )

    results = retriever.search(
        query,
        top_k=5
    )

    assert len(results) == 5

    for result in results:

        assert "chunk_id" in result
        assert "score" in result
        assert "rank" in result

        print(
            f"Rank {result['rank']} | "
            f"Chunk {result['chunk_id']} | "
            f"Score {result['score']:.4f}"
        )


if __name__ == "__main__":
    test_dense_retriever()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Rank 1 | Chunk 243 | Score 0.7075
Rank 2 | Chunk 175 | Score 0.7068
Rank 3 | Chunk 289 | Score 0.7043
Rank 4 | Chunk 241 | Score 0.7038
Rank 5 | Chunk 205 | Score 0.7028


# test_bm25_retriever.py

In [2]:
import sys

sys.path.insert(
    0,
    r"C:\Users\User\Nasa Rag SYS\rag\retrieval"
)

from bm25_retriever import BM25Retriever


def test_bm25_retriever():

    retriever = BM25Retriever()

    query = (
        "How does oxygen concentration affect "
        "flame behavior in microgravity?"
    )

    results = retriever.search(
        query,
        top_k=5
    )

    assert len(results) == 5

    for result in results:

        assert "chunk_id" in result
        assert "score" in result
        assert "rank" in result

        print(
            f"Rank {result['rank']} | "
            f"Chunk {result['chunk_id']} | "
            f"Score {result['score']:.4f}"
        )


if __name__ == "__main__":
    test_bm25_retriever()

Rank 1 | Chunk 15 | Score 2.2567
Rank 2 | Chunk 31 | Score 2.2567
Rank 3 | Chunk 35 | Score 2.2567
Rank 4 | Chunk 43 | Score 2.2567
Rank 5 | Chunk 91 | Score 2.2567


# test_hybrid_retriever.py

In [3]:
import sys

sys.path.insert(
    0,
    r"C:\Users\User\Nasa Rag SYS\rag\retrieval"
)

from hybrid_retriever import HybridRetriever


def test_hybrid_retriever():

    retriever = HybridRetriever()


# test_hybrid_reranker

In [4]:
import sys
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\User\Nasa Rag SYS"
)

VECTOR_STORE_DIR = PROJECT_ROOT / "rag" / "vector_store"
RETRIEVAL_DIR = PROJECT_ROOT / "rag" / "retrieval"
RAG_DIR = PROJECT_ROOT / "rag"


for path in [
    PROJECT_ROOT,
    RAG_DIR,
    VECTOR_STORE_DIR,
    RETRIEVAL_DIR
]:

    path = str(path)

    if path not in sys.path:
        sys.path.insert(0, path)


# ============================================================
# IMPORTS
# ============================================================

from hybrid_retriever import HybridRetriever
from Metadata_filter import MetadataFilter
from reranker import CrossEncoderReranker


# ============================================================
# CHUNKS
# ============================================================

CHUNKS_FILE = Path(
    r"C:\Users\User\Nasa Rag SYS\rag\embeddings"
    r"\processed\chunks\chunks.json"
)


# ============================================================
# TEST
# ============================================================

def test_hybrid_plus_reranker():

    query = (
        "How does oxygen concentration affect "
        "flame behavior in microgravity?"
    )

    print("\n")
    print("=" * 70)
    print("HYBRID RETRIEVAL + CROSS-ENCODER RERANKING TEST")
    print("=" * 70)

    print("\nQuery:")
    print(query)


    # ========================================================
    # STEP 1 — HYBRID RETRIEVAL
    # ========================================================

    print("\n" + "=" * 70)
    print("STEP 1: HYBRID RETRIEVAL")
    print("=" * 70)

    hybrid = HybridRetriever()

    hybrid_results = hybrid.search(
        query=query,
        dense_top_k=10,
        bm25_top_k=10,
        top_k=20
    )

    print(
        f"\nRetrieved {len(hybrid_results)} "
        f"hybrid candidates."
    )


    # ========================================================
    # DISPLAY HYBRID RESULTS
    # ========================================================

    for result in hybrid_results[:5]:

        print("\n" + "-" * 50)

        print(
            f"Chunk ID: "
            f"{result.get('chunk_id', 'N/A')}"
        )

        print(
            f"Hybrid Rank: "
            f"{result.get('hybrid_rank', 'N/A')}"
        )

        print(
            f"RRF Score: "
            f"{result.get('rrf_score', 0.0):.6f}"
        )

        print(
            f"Dense Rank: "
            f"{result.get('dense_rank', 'N/A')}"
        )

        print(
            f"Dense Score: "
            f"{result.get('dense_score', 'N/A')}"
        )

        print(
            f"BM25 Rank: "
            f"{result.get('bm25_rank', 'N/A')}"
        )

        print(
            f"BM25 Score: "
            f"{result.get('bm25_score', 'N/A')}"
        )


    # ========================================================
    # CHECK HYBRID RESULT STRUCTURE
    # ========================================================

    print("\n" + "=" * 70)
    print("HYBRID RESULT STRUCTURE")
    print("=" * 70)

    if hybrid_results:

        print(
            hybrid_results[0].keys()
        )


    # ========================================================
    # STEP 2 — METADATA ENRICHMENT
    # ========================================================

    print("\n" + "=" * 70)
    print("STEP 2: ENRICH RESULTS WITH CHUNK TEXT + METADATA")
    print("=" * 70)

    metadata_filter = MetadataFilter(
        CHUNKS_FILE
    )

    enriched_results = (
        metadata_filter.enrich_results(
            hybrid_results
        )
    )

    print(
        f"\nEnriched candidates: "
        f"{len(enriched_results)}"
    )


    # ========================================================
    # CHECK ENRICHED RESULT STRUCTURE
    # ========================================================

    print("\n" + "=" * 70)
    print("ENRICHED RESULT STRUCTURE")
    print("=" * 70)

    if enriched_results:

        print(
            enriched_results[0].keys()
        )


    # ========================================================
    # CHECK CHUNK TEXT
    # ========================================================

    print("\n" + "=" * 70)
    print("FIRST ENRICHED CHUNK")
    print("=" * 70)

    if enriched_results:

        first_result = enriched_results[0]

        print(
            f"\nChunk ID: "
            f"{first_result.get('chunk_id', 'N/A')}"
        )

        print(
            f"Hybrid Rank: "
            f"{first_result.get('hybrid_rank', 'N/A')}"
        )

        print("\nChunk Text:")

        print(
            first_result.get(
                "text",
                "TEXT NOT FOUND"
            )[:500]
        )

    else:

        print("No enriched results.")


    # ========================================================
    # STEP 3 — CROSS-ENCODER RERANKING
    # ========================================================

    print("\n" + "=" * 70)
    print("STEP 3: CROSS-ENCODER RERANKING")
    print("=" * 70)

    reranker = CrossEncoderReranker()

    reranked_results = reranker.rerank(
        query=query,
        candidates=enriched_results,
        top_k=5
    )

    print(
        f"\nReranked candidates: "
        f"{len(reranked_results)}"
    )


    # ========================================================
    # CHECK RERANKER RESULT STRUCTURE
    # ========================================================

    print("\n" + "=" * 70)
    print("RERANKER RESULT STRUCTURE")
    print("=" * 70)

    if reranked_results:

        print(
            reranked_results[0].keys()
        )


    # ========================================================
    # STEP 4 — FINAL RESULTS
    # ========================================================

    print("\n" + "=" * 70)
    print("FINAL RERANKED RESULTS")
    print("=" * 70)


    for result in reranked_results:

        print("\n" + "-" * 70)

        print(
            f"Final Reranker Rank: "
            f"{result.get('reranker_rank', 'N/A')}"
        )

        print(
            f"Chunk ID: "
            f"{result.get('chunk_id', 'N/A')}"
        )

        print(
            f"Reranker Score: "
            f"{result.get('reranker_score', 0.0):.4f}"
        )

        print(
            f"Original Hybrid Rank: "
            f"{result.get('hybrid_rank', 'N/A')}"
        )

        print(
            f"RRF Score: "
            f"{result.get('rrf_score', 0.0):.6f}"
        )

        print(
            f"Dense Rank: "
            f"{result.get('dense_rank', 'N/A')}"
        )

        print(
            f"BM25 Rank: "
            f"{result.get('bm25_rank', 'N/A')}"
        )

        print("\nChunk Text:")

        print(
            result.get(
                "text",
                "TEXT NOT FOUND"
            )[:500]
        )


    # ========================================================
    # FINAL STATUS
    # ========================================================

    print("\n" + "=" * 70)
    print("PIPELINE TEST COMPLETE")
    print("=" * 70)

    if reranked_results:

        print("SUCCESS")
        print(
            f"Final candidates: "
            f"{len(reranked_results)}"
        )

    else:

        print("NO RESULTS")


# ============================================================
# RUN TEST
# ============================================================

if __name__ == "__main__":

    test_hybrid_plus_reranker()



HYBRID RETRIEVAL + CROSS-ENCODER RERANKING TEST

Query:
How does oxygen concentration affect flame behavior in microgravity?

STEP 1: HYBRID RETRIEVAL


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Retrieved 20 hybrid candidates.

--------------------------------------------------
Chunk ID: 243
Hybrid Rank: 1
RRF Score: 0.016393
Dense Rank: 1
Dense Score: 0.7074596285820007
BM25 Rank: None
BM25 Score: None

--------------------------------------------------
Chunk ID: 15
Hybrid Rank: 2
RRF Score: 0.016393
Dense Rank: None
Dense Score: None
BM25 Rank: 1
BM25 Score: 2.2567201934900503

--------------------------------------------------
Chunk ID: 175
Hybrid Rank: 3
RRF Score: 0.016129
Dense Rank: 2
Dense Score: 0.7067693471908569
BM25 Rank: None
BM25 Score: None

--------------------------------------------------
Chunk ID: 31
Hybrid Rank: 4
RRF Score: 0.016129
Dense Rank: None
Dense Score: None
BM25 Rank: 2
BM25 Score: 2.2567201934900503

--------------------------------------------------
Chunk ID: 289
Hybrid Rank: 5
RRF Score: 0.015873
Dense Rank: 3
Dense Score: 0.7043361067771912
BM25 Rank: None
BM25 Score: None

HYBRID RESULT STRUCTURE
dict_keys(['chunk_id', 'rrf_score', 'dense_r

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded successfully.

CROSS-ENCODER RERANKING
Query: How does oxygen concentration affect flame behavior in microgravity?
Candidates received: 20
Generating reranker scores...

RERANKING COMPLETE
Top K: 5

--------------------------------------------------
Reranker Rank: 1
Chunk ID: 143
Reranker Score: -2.8965
Hybrid Rank: 14
Dense Rank: None
BM25 Rank: 7

--------------------------------------------------
Reranker Rank: 2
Chunk ID: 191
Reranker Score: -3.0640
Hybrid Rank: 20
Dense Rank: None
BM25 Rank: 10

--------------------------------------------------
Reranker Rank: 3
Chunk ID: 151
Reranker Score: -3.0825
Hybrid Rank: 16
Dense Rank: None
BM25 Rank: 8

--------------------------------------------------
Reranker Rank: 4
Chunk ID: 31
Reranker Score: -3.1579
Hybrid Rank: 4
Dense Rank: None
BM25 Rank: 2

--------------------------------------------------
Reranker Rank: 5
Chunk ID: 43
Reranker Score: -3.1874
Hybrid Rank: 8
Dense Rank: None
BM25 Rank: 4

Reranked candidates: 5

RE